# Solution Key — The `collections` Module
## All Labs

This notebook is **self-contained**: the file-based labs write their own small sample files first, so everything runs top-to-bottom with no external uploads. Each lab has a verified sample run and a short grading note.

---
## Lab 1 — Default Dictionary of Lists

Read a file of `word count` lines and build a `defaultdict(list)` mapping each word to the list of all its counts.

In [ ]:
# Write the sample file from the lab.
with open('counts.txt', 'w') as f:
    f.write('apple 2\npear 3\ncherry 5\napple 3\npear 6\napple 1\n')

from collections import defaultdict


def group_counts(filename):
    """Map each word to the list of counts that appear for it in the file."""
    result = defaultdict(list)
    with open(filename) as f:
        for line in f:
            word, count = line.split()
            result[word].append(count)   # counts kept as strings, matching the lab example
    return result


group_counts('counts.txt')

> **Note:** the win of `defaultdict(list)` is that `result[word].append(...)` works even the first time a word is seen — no `if word not in result` guard. The lab example keeps counts as strings; converting to `int` is a reasonable variation if a student asks.

---
## Lab 2 — Roll Your Own `MyDefaultDict`

Re-create the behavior **without** `collections`. Inherit from `dict`; the hook Python calls for a missing key inside `dict.__getitem__` is `__missing__`.

In [ ]:
class MyDefaultDict(dict):
    """A minimal defaultdict: inherit from dict, supply __missing__."""
    def __init__(self, default_factory, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.default_factory = default_factory

    def __missing__(self, key):
        # dict.__getitem__ calls this when the key is absent.
        value = self.default_factory()   # e.g. int() -> 0, list() -> []
        self[key] = value
        return value


counts = MyDefaultDict(int)
for letter in 'banana':
    counts[letter] += 1
print(dict(counts))

groups = MyDefaultDict(list)
groups['fruit'].append('apple')
print(dict(groups))

> **Why `__missing__` and not a full `__getitem__`?** `dict.__getitem__` already does the right thing and calls `self.__missing__(key)` only when the key is absent — so overriding `__missing__` is less code and avoids breaking normal lookups. A student who overrides `__getitem__` directly must remember to call `super().__getitem__` for keys that *do* exist; accept it if correct. Inheriting from `dict` is the expected answer to "what should it inherit from?"

---
## Lab 3 — `tail` with a Deque

Print the last *n* lines of a file. A `deque(maxlen=n)` fed the file object keeps exactly the last *n* lines automatically — older lines fall off the left as new ones arrive.

In [ ]:
# Write a 20-line sample file.
with open('log.txt', 'w') as f:
    for i in range(1, 21):
        f.write(f'line {i}\n')

from collections import deque


def tail(filename, n=10):
    """Print the last n lines of a file, like `tail -n`."""
    with open(filename) as f:
        last_lines = deque(f, maxlen=n)   # iterating the file yields lines
    for line in last_lines:
        print(line, end='')


tail('log.txt', 5)

> **The elegant part:** `deque(f, maxlen=n)` consumes the whole file but only ever holds *n* lines in memory — so it works on huge files.

---
## Lab 4 — A Deck of `Card` Named Tuples

Build all 52 cards. Looping suits on the outside and ranks on the inside yields the lab's order (all clubs 2→A, then diamonds, hearts, spades).

In [ ]:
from collections import namedtuple

Card = namedtuple('Card', 'rank suit')

RANKS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 'J', 'Q', 'K', 'A']
SUITS = ['clubs', 'diamonds', 'hearts', 'spades']

deck = [Card(rank, suit) for suit in SUITS for rank in RANKS]

print('cards in deck:', len(deck))
print('first 3 :', deck[:3])
print('last 3  :', deck[-3:])

> **Note:** 52 cards, no duplicates. The comprehension order matters for matching the example (`suit` outer, `rank` inner). Any rank representation is fine as long as the deck is complete; using `range(2, 11)` plus the face cards is the usual approach.

---
## Lab 5 — Word Count with `Counter`

Read a file, split into words, and report the 10 most common. `Counter` plus `most_common` makes this nearly a one-liner.

In [ ]:
# Write a small sample text.
sample = ('the quick brown fox the lazy dog the fox ran '
          'the dog sat the cat the cat the cat sat down')
with open('story.txt', 'w') as f:
    f.write(sample)

from collections import Counter


def top_words(filename, n=10):
    """Return the n most common words in a file as (word, count) pairs."""
    with open(filename) as f:
        words = f.read().split()
    return Counter(words).most_common(n)


for word, count in top_words('story.txt'):
    print(f'{word:>8} : {count}')

> **Note:** `Counter(words).most_common(n)` is the target. For real text you'd usually lower-case and strip punctuation first (as in the Week 1 files lab); note it, but the bare version satisfies this lab. `most_common()` with no argument returns *all* words, ranked.